In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import datetime
import warnings
import os
import json
import joblib
warnings.filterwarnings('ignore')


In [2]:
data = pd.read_csv('../data/ev_battery_degradation_v1.csv')
df = data.copy()

In [3]:
data.head(10)

,Vehicle_ID,Car_Model,Battery_Type,Battery_Capacity_kWh,Vehicle_Age_Months,Total_Charging_Cycles,Avg_Temperature_C,Fast_Charge_Ratio,Avg_Discharge_Rate_C,Driving_Style,Internal_Resistance_Ohm,SoH_Percent,Battery_Status
0,1fb46ae8,Tesla Model 3,NMC,75.0,41,390,21.5,0.51,2.22,Aggressive,0.0362,94.60,Healthy
1,b7ef35aa,Tesla Model 3,NMC,75.0,29,401,18.0,0.62,1.34,Aggressive,0.0333,95.68,Healthy
2,76cb49e0,Ford Mustang Mach-E,NMC,88.0,71,941,18.4,0.78,1.48,Conservative,0.0526,89.80,Healthy
3,456a7aef,Ford Mustang Mach-E,NMC,88.0,57,378,10.8,0.61,0.72,Moderate,0.0314,96.29,Healthy
4,bd758049,Tesla Model 3,NMC,75.0,58,239,30.3,0.89,1.48,Conservative,0.0297,96.75,Healthy
5,d5f95b58,Hyundai Ioniq 5,NMC,72.6,27,225,20.0,0.35,2.50,Moderate,0.0265,98.07,Healthy
6,b135d980,Hyundai Ioniq 5,NMC,72.6,49,734,3.2,0.51,0.65,Conservative,0.0401,94.20,Healthy
7,a99c510d,Hyundai Ioniq 5,NMC,72.6,12,121,27.4,0.29,1.41,Moderate,0.0223,98.76,Healthy
8,9d628060,Ford Mustang Mach-E,NMC,88.0,40,170,23.7,0.71,1.23,Conservative,0.0236,98.53,Healthy
9,c06175d3,Wuling Air EV,LFP,26.0,12,100,19.6,0.59,1.50,Conservative,0.0208,100.00,Healthy


In [4]:
df.isnull().sum()

Vehicle_ID                 0
Car_Model                  0
Battery_Type               0
Battery_Capacity_kWh       0
Vehicle_Age_Months         0
Total_Charging_Cycles      0
Avg_Temperature_C          0
Fast_Charge_Ratio          0
Avg_Discharge_Rate_C       0
Driving_Style              0
Internal_Resistance_Ohm    0
SoH_Percent                0
Battery_Status             0
dtype: int64

In [5]:
df2=df.copy()



In [6]:
df.columns

Index(['Vehicle_ID', 'Car_Model', 'Battery_Type', 'Battery_Capacity_kWh',
       'Vehicle_Age_Months', 'Total_Charging_Cycles', 'Avg_Temperature_C',
       'Fast_Charge_Ratio', 'Avg_Discharge_Rate_C', 'Driving_Style',
       'Internal_Resistance_Ohm', 'SoH_Percent', 'Battery_Status'],
      dtype='str')

In [7]:
# 0) Minimal cleaning (recommended)
df2["Vehicle_ID"] = df2["Vehicle_ID"].astype("string").str.strip()
df2["Battery_Status"] = df2["Battery_Status"].astype("string").str.strip().str.title()

df2["SoH_Percent"] = pd.to_numeric(df2["SoH_Percent"], errors="coerce")

# Keep only rows where KPI targets exist
kpi_df = df2.dropna(subset=["Vehicle_ID", "SoH_Percent", "Battery_Status"])

# Thresholds (you can change later)
LOW_SOH_TH = 70
HIGH_SOH_TH = 90

# 1) Total Vehicles = count of unique Vehicle_ID
total_vehicles = kpi_df["Vehicle_ID"].nunique()

# 2) Average SoH (%)
avg_soh = kpi_df["SoH_Percent"].mean()

# 3) Median SoH (%)
median_soh = kpi_df["SoH_Percent"].median()

# 4) SoH Std Dev
soh_std = kpi_df["SoH_Percent"].std()

# 5) % Healthy / % Degraded / % Critical
status_pct = (
    kpi_df["Battery_Status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
healthy_pct  = float(status_pct.get("Healthy", 0.0))
degraded_pct = float(status_pct.get("Degraded", 0.0))
critical_pct = float(status_pct.get("Critical", 0.0))

# 6) Low-SoH Share = % vehicles with SoH < 70
low_soh_share_pct = (kpi_df["SoH_Percent"] < LOW_SOH_TH).mean() * 100

# 7) High-SoH Share = % vehicles with SoH >= 90
high_soh_share_pct = (kpi_df["SoH_Percent"] >= HIGH_SOH_TH).mean() * 100

In [8]:
# Average Charging Cycles
avg_cycles = df["Total_Charging_Cycles"].mean()

# Median Charging Cycles
median_cycles = df["Total_Charging_Cycles"].median()

print(f"Average Charging Cycles: {avg_cycles:.2f}")
print(f"Median Charging Cycles: {median_cycles:.2f}")

Average Charging Cycles: 457.02
Median Charging Cycles: 400.00


In [9]:
# Average Temperature
avg_temp = df["Avg_Temperature_C"].mean()

# Hot Operation Share (temperature >= 35C)
hot_operation_share = (df["Avg_Temperature_C"] >= 35).mean() * 100

print(f"Average Temperature: {avg_temp:.2f} °C")
print(f"Hot Operation Share: {hot_operation_share:.2f}%")

Average Temperature: 25.07 °C
Hot Operation Share: 10.78%


In [10]:
# Average Fast Charge Ratio
avg_fast_charge = df["Fast_Charge_Ratio"].mean()

# High Fast Charge Share (>= 0.5)
high_fast_charge_share = (df["Fast_Charge_Ratio"] >= 0.5).mean() * 100

print(f"Average Fast Charge Ratio: {avg_fast_charge:.2f}")
print(f"High Fast Charge Share: {high_fast_charge_share:.2f}%")

Average Fast Charge Ratio: 0.50
High Fast Charge Share: 49.69%


In [11]:
# Average Internal Resistance
avg_internal_resistance = df["Internal_Resistance_Ohm"].mean()

# P90 threshold
p90_resistance = df["Internal_Resistance_Ohm"].quantile(0.90)

# Share above P90
high_resistance_share = (df["Internal_Resistance_Ohm"] >= p90_resistance).mean() * 100

print(f"Average Internal Resistance: {avg_internal_resistance:.5f} Ohm")
print(f"P90 Internal Resistance Threshold: {p90_resistance:.5f}")
print(f"High Internal Resistance Share: {high_resistance_share:.2f}%")

Average Internal Resistance: 0.03136 Ohm
P90 Internal Resistance Threshold: 0.04520
High Internal Resistance Share: 10.12%


In [12]:
avg_discharge_rate = df["Avg_Discharge_Rate_C"].mean()

print(f"Average Discharge Rate (C): {avg_discharge_rate:.2f}")

Average Discharge Rate (C): 1.49


In [13]:
aggressive_driving_share = (
    df["Driving_Style"].str.lower() == "aggressive"
).mean() * 100

print(f"Aggressive Driving Share: {aggressive_driving_share:.2f}%")

Aggressive Driving Share: 19.39%


In [14]:
cycles_norm = df["Total_Charging_Cycles"] / df["Total_Charging_Cycles"].max()
temp_norm = df["Avg_Temperature_C"] / df["Avg_Temperature_C"].max()
fastcharge_norm = df["Fast_Charge_Ratio"] / df["Fast_Charge_Ratio"].max()
resistance_norm = df["Internal_Resistance_Ohm"] / df["Internal_Resistance_Ohm"].max()

In [15]:
stress_score = (
    0.35 * cycles_norm +
    0.25 * temp_norm +
    0.25 * fastcharge_norm +
    0.15 * resistance_norm
)

In [16]:
remaining_life_index = (1 - stress_score) * 100

df["Remaining_Life_Index"] = remaining_life_index

In [17]:
avg_remaining_life = df["Remaining_Life_Index"].mean()

print(f"Average Remaining Life Index: {avg_remaining_life:.2f}")

Average Remaining Life Index: 60.09


In [18]:
kpi_payload = {

    "fleet_health_kpis": {

        "total_vehicles": int(total_vehicles),

        "average_soh_percent": round(avg_soh, 2),
        "median_soh_percent": round(median_soh, 2),
        "soh_std_dev": round(soh_std, 2),

        "healthy_percent": round(healthy_pct, 2),
        "degraded_percent": round(degraded_pct, 2),
        "critical_percent": round(critical_pct, 2),

        "low_soh_share_percent": round(low_soh_share_pct, 2),
        "high_soh_share_percent": round(high_soh_share_pct, 2)
    },


    "degradation_driver_kpis": {

        "avg_charging_cycles": round(avg_cycles, 2),
        "median_charging_cycles": round(median_cycles, 2),

        "avg_temperature_c": round(avg_temp, 2),
        "hot_operation_share_percent": round(hot_operation_share, 2),

        "avg_fast_charge_ratio": round(avg_fast_charge, 2),
        "high_fast_charge_share_percent": round(high_fast_charge_share, 2),

        "avg_internal_resistance_ohm": round(avg_internal_resistance, 5),
        "high_internal_resistance_share_percent": round(high_resistance_share, 2)
    },


    "operational_profile_kpis": {

        "avg_discharge_rate_c": round(avg_discharge_rate, 2),

        "aggressive_driving_share_percent": round(aggressive_driving_share, 2),

        "avg_remaining_life_index": round(avg_remaining_life, 2)
    }

}

In [21]:
dashboard_path = "../artifacts/dashboard"

os.makedirs(dashboard_path, exist_ok=True)  

payload_file = os.path.join(dashboard_path, "kpi_payload.json")

with open(payload_file, "w") as f:
    json.dump(kpi_payload, f, indent=4)

print("KPI payload saved to:", payload_file)

KPI payload saved to: ../artifacts/dashboard\kpi_payload.json
